# Walks through the sensor graph: which routes a prediction took

**Explanation notebook 3 in `notebooks/xai/`.** `notebooks/README.md` says how every notebook is laid out, and `notebooks/xai/README.md` says what every explanation notebook shares.

## What this notebook is for

The network's graph layers pass numbers between connected sensors, so a prediction is not built from each sensor on its own: part of what a sensor holds arrived from its neighbours, and from theirs. **GNN-LRP** (Schnake et al., 2022) is an explanation method made for exactly that kind of network. It reads the trained network's own weights and splits one prediction exactly into shares, one per route through the sensor graph, so it can say which **connections** carried a prediction.

This notebook applies it as its authors define it: their rule, their setting of it, and their own check of the result. It is **not in the paper** this project re-implements (assumption A-29), and nothing in it is taken from that paper's explanation method or its results.

## What it does not do

- **It does not train anything.** It loads the same stored network every explanation notebook here explains.
- **It does not say whether the paper's graph is good.** It reads one trained network. Whether the graph helps needs networks trained on other graphs (assumption A-27).
- **It does not give cycles.** Its shares are parts of the network's score before its last step, for reasons section 1 gives.
- **It does not say what wears an engine.** A route's share describes how this network computed, not how an engine fails.

## The notebook in one list

| # | Step | What goes in | What comes out |
|---|------|--------------|----------------|
| 1 | **Load** the trained network | the stored five-seed run | seed 42's network and its data, checked against the run |
| 2 | **Split one window into walks** | the last window of the held-back engine with the most windows | 38,416 walk shares, checked, and the largest |
| 3 | **Follow the same engine's life** | every window of that engine | what stays and what moves, and what flowed along each connection |
| 4 | **Run the authors' check** | the same windows | the score as sensors are added and removed in the walks' order, beside simpler and random orders |
| 5 | **Set flow beside removal** | the same windows | what removing each connection does, which is a different question |
| 6 | **Read the five-seed measurement** | the stored `walks/` folders | trained networks beside their own untrained starting points |

## 1. Background

### What one graph layer does

Each of the network's three graph layers gives every sensor a new set of numbers made from its neighbours' current numbers:

```
new_i = ReLU( sum over sensors j of  A_hat[i, j] * old_j * W )
```

`A_hat[i, j]` is zero unless sensor `j` is joined to sensor `i` in the paper's graph or is `i` itself; each row adds up to 1. `W` is the layer's learned weight table, and `ReLU` turns anything negative into zero. After one layer a sensor holds a mix of its neighbours; after three, readings from up to three steps away.

### What a walk is

A **walk** is the list of sensors one piece of a reading passes through, one per layer: `T50 -> T30 -> T30 -> Ps30` is T50's reading, carried into T30 by the first layer, kept at T30 by the second, carried into Ps30 by the third. With 14 sensors and three layers there are 14 × 14 × 14 × 14 = 38,416 possible walks; the paper's graph allows 5,178 of them, and the rest carry exactly zero.

### Why one prediction splits exactly into walks

The network's last step turns a number, the **score**, into a prediction with `125 / (1 + e^-score)`, the sigmoid. For one window, every ReLU is either on (passes its input) or off (passes zero). Holding that pattern, everything between the numbers entering the graph layers and the score is multiplication and addition, so the score multiplies out into a sum of products, one per walk, plus a **constant** from the network's two last layers that belongs to no sensor.

The sigmoid itself bends, so shares of the score cannot be turned into cycles that still add up. The shares are therefore **parts of the score, not cycles**: a score of 0 is a prediction of 62.5 cycles, higher scores predict more.

### The rule and its setting γ

At each graph layer, the share a sensor received is handed back to the sensors it came from in proportion to what each sent. The authors count positive contributions more heavily by a setting γ: with γ = 0 every contribution counts as it was (their simpler baseline, GNN-GI); a positive γ shrinks the shares that pulled against the result. This notebook uses **their own setting**: their published code's default, 3 at the first graph layer falling to 0 at the last, with γ = 0 at the first layer because their rule needs the numbers entering a layer to be zero or more and this network's first layer receives rescaled readings, which can be negative — the same fall-back the authors use themselves. For three layers that is 0, 1.5 and 0 (assumption A-29).

### A small example

`explain/README.md` works one through by hand on three real sensors: in a toy network with one graph layer, test engine 5's T50 reading of 0.8176 sends `0.8176 × 1/3 = 0.2725` of the score along the walk T50 -> T30, and five such walks plus the constant 0.5 give exactly the score 0.2202. `tests/test_examples_torch.py` checks every number.

### Three groups the graph cannot tell apart

In the paper's graph, P15, Nf, NRf and BPR are joined to exactly the same sensors, and so are P30 and Ps30, and Nc and W31 (finding F-18). From the first graph layer on, each group's members hold the same numbers, and where a walk ends among them is decided by the network's random starting weights. This notebook counts each group as one sensor wherever a walk ends.

## 2. Setup

The same stored run every explanation notebook here uses: the most recent five-seed run of `config/fd001.yaml` with the paper's graph, the last-cycle temporal stage and the network sizes of Table 4, whose trained networks were saved.

In [ ]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import torch
except ImportError as missing:
    raise ImportError(
        "This notebook runs a trained network, which needs PyTorch, and PyTorch is "
        'not installed. Install the project with  pip install -e ".[model]"'
    ) from missing

from thesis_gnn_rul_xai.config import load_config
from thesis_gnn_rul_xai.explain import (
    aufc,
    connection_relevance,
    connection_removal,
    connections_of,
    conservation_error,
    crossing_relevance,
    indistinguishable_sensors,
    node_flipping,
    node_relevance,
    paper_gammas,
    subgraph_scores,
    top_walks,
    walk_relevance,
)
from thesis_gnn_rul_xai.graph.adjacency import to_networkx
from thesis_gnn_rul_xai.models import ModelConfig
from thesis_gnn_rul_xai.training import evaluate_model, find_runs, load_seed
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging

configure_logging(logging.ERROR)
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch     : {torch.__version__}")

In [ ]:
config = load_config(PROJECT_ROOT / "config" / "fd001.yaml")
runs = [
    run for run in find_runs(
        preprocessing=config, seeds=5, with_weights=True, split="varies_with_seed",
        early_stopping=True, lr_schedule="step", temporal_stage="last_cycle",
    )
    if run.model_config == ModelConfig()
]
if not runs:
    raise FileNotFoundError(
        "reports/ holds no five-seed run of config/fd001.yaml that saved its trained "
        "networks. Make one with\n\n"
        "    python scripts/train.py --config config/fd001.yaml --seeds 5\n\n"
        "and run this notebook again."
    )
run = runs[-1]
SEED = 42
print(f"Stored run : {run.name}")
print(f"Commit     : {run.summary['commit']}")

## 3. The trained network

### Step 1 — Load the trained network

**What.** Load seed 42's network and rebuild its data.

**Check.** The loaded network must score the test engines exactly as the run recorded.

In [ ]:
model, prepared = load_seed(run, SEED)
sensors = list(prepared.sensor_symbols)
groups = indistinguishable_sensors(model, sensors)
GAMMA = paper_gammas(len(model.graph_layers))

recorded = float(run.results.set_index("seed").loc[SEED, "rmse"])
scored = evaluate_model(model, prepared.test)["rmse"]
same_network = bool(np.isclose(scored, recorded, rtol=0, atol=1e-6))
assert same_network
print(f"OK  test RMSE {scored:.6f} cycles, as the run recorded for seed {SEED}.")
print("    sensors the graph cannot tell apart:", "; ".join(", ".join(g) for g in groups))
print("    the authors' setting of gamma, graph layer by graph layer:", GAMMA)

## 4. The explanation, step by step

### Step 2 — One window, split into walks

**What.** Take the last window of the held-back engine with the most windows, close to that engine's failure, and split its score into walks with the authors' setting.

**Why.** One window is small enough to check completely: the shares must add up to the score, and the score must turn into the network's own prediction.

In [ ]:
held_back = prepared.validation
units, counts = np.unique(held_back.units, return_counts=True)
engine = int(units[np.argmax(counts)])
engine_rows = np.flatnonzero(held_back.units == engine)
engine_rows = engine_rows[np.argsort(held_back.end_cycles[engine_rows])]
window = held_back.features[engine_rows[-1]]

result = walk_relevance(model, window)
assert list(result.gamma) == GAMMA
walks_sum = float(result.relevance.sum())
print(f"Engine {engine}, window ending at cycle {held_back.end_cycles[engine_rows[-1]]}")
print(f"  prediction                       {result.predicted[0]:9.3f} cycles")
print(f"  score before the sigmoid         {result.score[0]:+9.4f}")
print(f"  walks added up                   {walks_sum:+9.4f}")
print(f"  constant, belonging to no sensor {result.constant[0]:+9.4f}")
print(f"  walks + constant - score         {walks_sum + result.constant[0] - result.score[0]:+9.1e}")
print(f"  walks that are not zero          {int((result.relevance[0] != 0).sum()):,} of {result.relevance[0].size:,}")

**Check.** Walks plus constant must give back the score, and the score through the sigmoid must be the network's prediction. At γ = 0, the authors' GNN-GI, every share is also a number times how fast the score changes with it, so each sensor's total must equal that product computed by PyTorch's own differentiation, which shares no code with the walks; that is checked at γ = 0, where it must hold.

In [ ]:
adds_up = conservation_error(result) < 1e-9
turns_into_prediction = bool(np.isclose(125 / (1 + np.exp(-result.score[0])), result.predicted[0], atol=1e-3))

baseline_walks = walk_relevance(model, window, gamma=0.0)
double = load_seed(run, SEED)[0].double()
nodes = double.temporal(torch.from_numpy(window[None].astype(np.float64))).detach().requires_grad_(True)
hidden = nodes
for layer in double.graph_layers:
    hidden = torch.relu(layer(hidden, double.adjacency))
double.fc2(torch.relu(double.fc1(torch.flatten(hidden, start_dim=1)))).sum().backward()
by_gradient = (nodes.grad * nodes).sum(dim=2).detach().numpy()[0]
gi_matches_gradient = bool(np.allclose(node_relevance(baseline_walks)[0], by_gradient, atol=1e-9))
assert adds_up and turns_into_prediction and gi_matches_gradient
moved = np.abs(result.relevance - baseline_walks.relevance).sum() / np.abs(baseline_walks.relevance).sum()
print(f"OK  walks + constant = score, to {conservation_error(result):.1e}")
print("OK  the score through the sigmoid is the network's own prediction")
print("OK  at gamma 0, each sensor's total equals its number times the score's derivative")
print(f"    between gamma 0 and the authors' setting, the walks move by {moved:.1%} of their total size")

The twelve walks that carry the most on this window, with the groups the graph cannot tell apart counted as one wherever a walk ends, beside the size of every walk added up ignoring its sign:

In [ ]:
largest = top_walks(result, sensors, count=12, pool=groups)
display(largest.round(4))
parts = crossing_relevance(result, sensors, pool=groups)
ratio = float(parts["absolute_total"].iloc[0] / abs(parts["stays"].iloc[0] + parts["moves"].iloc[0]))
print(f"Every walk's size added up is {ratio:.1f} times the size of their sum on this window.")

#### Reading the table

On this window, close to the engine's failure (a prediction of 4.596 cycles), the four largest walks all end inside the group P15, Nf, NRf and BPR and all pull the score down: the walk starting at BPR by −0.5046, at T24 by −0.4696, at Nf by −0.2117 and at NRf by −0.1820, in score units. The group's members hold the same numbers from the first graph layer on, so these four are one route read from four starting readings. The first walk that reaches the other side of the graph comes twelfth: `Nc -> P30/Ps30 -> Nc/W31 -> W32`, −0.0526. Ignoring signs, the walks add up to 3.8 times the size of their sum on this window.

On this window the authors' setting and γ = 0 give the same walks (the change printed above is 0.0 %). It is one of the windows where every sensor's input to the first graph layer has the same sign, the one case where γ cannot change a share; step 4 counts how often that happens.

### Step 3 — The same engine's whole life

**What.** Split every window of the engine into walks, and follow three things over its life: the walks that stay within their starting sensor's group, the walks that move between groups, and the constant; and, on the sensor graph, what flowed along each connection, averaged over the whole life.

In [ ]:
life = walk_relevance(model, held_back.features[engine_rows])
life_parts = crossing_relevance(life, sensors, pool=groups)
cycles = held_back.end_cycles[engine_rows]

figure, axis = plt.subplots(figsize=(8, 3.2))
axis.plot(cycles, life_parts["stays"], label="walks that stay within their sensor's group")
axis.plot(cycles, life_parts["moves"], label="walks that move between groups")
axis.plot(cycles, life_parts["constant"], label="constant, belonging to no sensor")
axis.plot(cycles, life_parts["score"], "k--", label="score (the three added up)")
axis.axhline(0, color="0.6", linewidth=0.6)
axis.set_xlabel("cycle the window ends at")
axis.set_ylabel("share of the score before the sigmoid")
axis.legend(fontsize=7, loc="lower left")
axis.set_title(f"Engine {engine}: how its score splits, window by window")
plt.show()
print(f"Largest miss of stays + moves + constant against the score: "
      f"{np.abs(life_parts['stays'] + life_parts['moves'] + life_parts['constant'] - life_parts['score']).max():.1e}")
thirds = np.array_split(np.arange(len(cycles)), 3)
display(pd.DataFrame({
    name: [life_parts[column].iloc[rows].mean() for rows in thirds]
    for name, column in (("stays", "stays"), ("moves", "moves"), ("constant", "constant"),
                         ("score", "score"), ("prediction, cycles", "predicted"))
}, index=["first third", "middle third", "last third"]).round(3))

In [ ]:
flow = connection_relevance(life, sensors, pool=groups).mean(axis=0)
graph = to_networkx(prepared.adjacency_raw, sensors)
place = nx.kamada_kawai_layout(graph)
pairs = connections_of(prepared.adjacency_raw)
values = np.array([flow[i, j] + flow[j, i] for i, j in pairs])
peak = np.abs(values).max()

figure, axis = plt.subplots(figsize=(7, 5.2))
for (i, j), value in zip(pairs, values):
    (x1, y1), (x2, y2) = place[sensors[i]], place[sensors[j]]
    axis.plot([x1, x2], [y1, y2], color="tab:blue" if value >= 0 else "tab:orange",
              linewidth=0.6 + 6 * abs(value) / peak, alpha=0.8, zorder=1)
for name, (x, y) in place.items():
    axis.scatter([x], [y], s=420, color="white", edgecolors="0.3", zorder=2)
    axis.text(x, y, name, ha="center", va="center", fontsize=7, zorder=3)
axis.set_title(f"Engine {engine}: what flowed along each connection, averaged over its life\n"
               "width = size; blue raised the prediction, orange lowered it")
axis.axis("off")
plt.show()
flows = pd.Series(values, index=[f"{sensors[i]}-{sensors[j]}" for i, j in pairs])
print("Largest average flows, both directions added, score units:")
display(flows.reindex(flows.abs().sort_values(ascending=False).index).head(8).round(4).to_frame("flow"))

#### Reading the figures

The score falls from 2.649 on average in the first third of the engine's life to −0.934 in the last third, and the prediction from 115.403 to 41.706 cycles. The constant hardly moves (0.887, 0.911, 0.835), so the fall comes from the walks, and mostly from walks that move between groups, which go from +0.581 to −1.616; walks that stay within a group go from +1.181 to −0.152.

On the graph, averaged over the whole life, the widest connection is T24–T30 (+0.1931), then the links inside the group P15, Nf, NRf and BPR (from +0.1424 to +0.1649) and T30–Ps30 (+0.1515); all eight of the largest raised the prediction on average. An average over a life mixes early windows, where shares are mostly positive, with late ones, where they turn negative, so a thin line can hide two large contributions of opposite sign.

### Step 4 — The authors' check: adding and removing sensors

**What.** The check the GNN-LRP paper uses on its own explanations (its Section 4.1, which it calls node flipping). For every window of the engine, sensors are added one at a time to an empty graph in the order the walks rate highest (their Equation 14), and removed one at a time from the full graph in the order the walks say matters least (their Equation 15). After each step the network is run on the graph that is left, a sensor being taken out by cutting all its connections, its link to itself included (assumption A-33). The paper's summary of each curve is its mean, the **area under the flipping curve**.

**Set beside it, as in the paper:** the same walks added up by their starting sensor, the authors' simpler GNN-GI and its starting sensors, and random orders.

**What a good ranking does.** It raises the score fastest while adding (higher is better) and moves it least while removing (lower is better).

In [ ]:
engine_windows = held_back.features[engine_rows]
life_gi = walk_relevance(model, engine_windows, gamma=0.0)
with torch.no_grad():
    mixed_in = (model.adjacency.double() @ model.temporal(torch.from_numpy(engine_windows)).double())[:, :, 0].numpy()
one_sign = (mixed_in > 0).all(axis=1) | (mixed_in < 0).all(axis=1)
moved_by = (np.abs(life.relevance - life_gi.relevance).reshape(len(engine_rows), -1).sum(axis=1)
            / np.abs(life_gi.relevance).reshape(len(engine_rows), -1).sum(axis=1))
print(f"On {int(one_sign.sum())} of {len(engine_rows)} windows every sensor's input to the first graph layer has one sign; "
      f"there the authors' setting and gamma 0 give identical walks (largest change {moved_by[one_sign].max():.1e}).")
print(f"On the other {int((~one_sign).sum())} the walks move by a median of {np.median(moved_by[~one_sign]):.1%} of their total size.")
flipping = node_flipping(model, engine_windows, {"GNN-LRP": life, "GNN-GI": life_gi})
summary = aufc(flipping)
display(summary.pivot(index="ranking", columns="task", values="mean").round(3))

figure, axes = plt.subplots(1, 2, figsize=(10, 3.4))
styles = {"GNN-LRP, walks": ("k", "-"), "GNN-LRP, starting sensors": ("k", "--"),
          "GNN-GI, walks": ("tab:cyan", "-"), "GNN-GI, starting sensors": ("tab:cyan", "--"),
          "random": ("0.6", ":")}
steps = [f"step_{k}" for k in range(len(sensors) + 1)]
for axis, task in zip(axes, ("activation", "pruning")):
    for ranking, (colour, line) in styles.items():
        mean_curve = flipping[(flipping.task == task) & (flipping.ranking == ranking)][steps].mean()
        axis.plot(range(len(steps)), mean_curve.to_numpy(), color=colour, linestyle=line, label=ranking)
    axis.set_xlabel("sensors added" if task == "activation" else "sensors removed")
    axis.set_ylabel("score before the sigmoid" if task == "activation" else "distance from the full graph's score")
    axis.set_title(f"{'Adding' if task == 'activation' else 'Removing'}: mean over {len(engine_rows)} windows")
axes[0].legend(fontsize=7)
plt.show()
full_graph_scores = bool(np.allclose(
    [subgraph_scores(model, w, np.ones((1, len(sensors)), dtype=bool))[0] for w in engine_windows[:5]],
    life.score[:5], atol=1e-9))
assert full_graph_scores
lrp = flipping[flipping.ranking == "GNN-LRP, walks"].set_index(["window", "task"])["aufc"]
rnd = flipping[flipping.ranking == "random"].set_index(["window", "task"])["aufc"]
for task in ("activation", "pruning"):
    better = (lrp.xs(task, level="task") > rnd.xs(task, level="task")) if task == "activation" else \
             (lrp.xs(task, level="task") < rnd.xs(task, level="task"))
    print(f"{task}: the walks beat the random orders on {int(better.sum())} of {len(better)} windows")

#### Reading the check

On 56 of the engine's 240 windows every sensor's input to the first graph layer has one sign, and there the authors' setting and γ = 0 give identical walks; on the other 184 the walks move by a median of 34.0 % of their total size.

By the authors' check, the walks describe this network well on this engine: they beat the random orders on 230 of 240 windows when adding and on 233 when removing, with mean areas of 1.534 against 1.084 when adding (higher is better) and 0.696 against 1.166 when removing (lower is better). Added up by starting sensor they do worse — 1.317 and 1.074 — so it is the routes, not only the starting sensors, that match the network's own output. The authors' setting and GNN-GI land close together (1.534 against 1.495 adding, 0.696 against 0.717 removing).

### Step 5 — What flowed along a connection, beside what removing it does

**What.** A different question from the authors' check: remove one connection at a time from the raw graph, rescale the graph as the network's own was made, and run the same network again (assumption A-30). This is not part of the GNN-LRP paper; it is set beside the walks because "the connection that matters most" can mean either what flowed along it or what removing it does.

In [ ]:
removal = connection_removal(model, engine_windows, prepared.adjacency_raw,
                             sensors, method=prepared.config.adjacency_normalization)
pooled = connection_relevance(life, sensors, pool=groups)
flow_size = np.abs(np.stack([pooled[:, i, j] + pooled[:, j, i] for i, j in pairs], axis=1)).mean(axis=0)
removal_size = removal.abs().mean(axis=0).to_numpy()
agreement = stats.spearmanr(flow_size, removal_size).statistic
print(f"Rank correlation between flow and removal over the {len(pairs)} connections: {agreement:.2f}")
print("Largest average removal effects, cycles:")
display(removal.abs().mean().sort_values(ascending=False).head(6).round(2).to_frame("removal"))

#### Reading the comparison

On this engine the two measures rank the 41 connections almost independently: a rank correlation of 0.04. Removing T24–T30 moves the prediction most, by 5.78 cycles on average, and it also carries the largest flow. But the next three largest removal effects — Nc–W32 (3.59), T30–T50 (3.54) and T50–Nc (3.47) — are on connections outside the eight largest flows. Removal also rescales the rest of both sensors' rows, so it measures something else (assumption A-30); over the five seeds' test windows the two even rank the connections in weakly opposite orders (step 6).

### Step 6 — The measurement over five seeds

**What.** The numbers above are one seed. `scripts/measure_walks.py` repeated the measurement on the last window of every test engine for all five saved seeds, and explained, beside each trained network, an untrained one built from the same seed exactly as training builds it — the trained network's own starting point. This cell reads its newest stored result rather than recomputing it.

**Why the untrained network.** Much of what a walk share says is fixed by the graph's own weights before any training: a sensor's connection to itself weighs one over its number of connections plus one. A figure says something about training only where the two differ.

In [ ]:
folders = sorted((run.path / "walks").glob("*/summary.json"))
if not folders:
    raise FileNotFoundError(
        f"{run.name} has no walks/ folder. Make one with\n\n"
        f"    python scripts/measure_walks.py --run {run.name}\n"
    )
measured = json.loads(folders[-1].read_text(encoding="utf-8"))
assert measured["gammas"]["authors"] is None, "the newest measurement predates the authors' setting"
per_seed = pd.read_csv(folders[-1].parent / "per_seed.csv")
print(f"Measured at {measured['commit']} on {measured['finished_utc']}, seeds {measured['seeds']}")
columns = {"stays_share_median": "walks staying within a group, share of size",
           "constant_share_median": "constant, share of the score's size",
           "cancellation_median": "walk sizes / size of their sum",
           "removal_agreement": "flow against removal, rank correlation",
           "aufc_activation_GNN-LRP, walks": "authors' check, adding: walks",
           "aufc_activation_random": "authors' check, adding: random",
           "aufc_pruning_GNN-LRP, walks": "authors' check, removing: walks",
           "aufc_pruning_random": "authors' check, removing: random"}
table = per_seed.groupby("network")[list(columns)].agg(["mean", "std"]).T.unstack()
table.index = [columns[i] for i in table.index]
display(table.round(3))
trained = per_seed[per_seed.network == "trained"].sort_values("seed")
for task in ("activation", "pruning"):
    p = stats.ttest_rel(trained[f"aufc_{task}_GNN-LRP, walks"], trained[f"aufc_{task}_random"]).pvalue
    print(f"Trained, {task}: walks against random orders, paired over seeds, p = {p:.4f}")

#### What the five seeds say

Over the five seeds, on the last window of every test engine: by the authors' check the walks beat the random orders on every seed, adding (1.408 ± 0.094 against 0.982 ± 0.162, paired p = 0.0014) and removing (0.645 ± 0.085 against 1.053 ± 0.100, p < 0.0001). Walks that stay within a group hold 6.4 ± 3.2 % of the walk size in the trained networks, against 16.6 ± 3.6 % in their untrained starting points; the constant is 51 ± 4 % of the score's size, against 82 ± 14 %; the walks cancel about equally before and after training (7.1 against 7.7); and flow and removal rank the connections in weakly opposite orders after training (−0.26 ± 0.14), where before training they agreed moderately (0.41 ± 0.26). The log records the same measurement for four other configurations (assumptions A-29, A-30 and A-33).

## 5. What walks do and do not show

1. **What object is explained?** One window's score before the sigmoid.
2. **What does the method have access to?** Every weight, the sensor graph, and which ReLUs are on at that window. It is specific to this kind of network.
3. **What does it change?** Nothing. No input is replaced and no connection is removed. (The authors' check, step 4, does remove sensors, to test the explanation.)
4. **What does it measure?** How the network's own arithmetic, at that window, divides the score among routes through the graph, under the authors' rule and setting.
5. **What is the result?** One share per walk — 38,416 per window, 5,178 of them possible on the paper's graph — which add up, with a constant no sensor owns, to the score.
6. **What does it assume?** That holding the ReLUs as the window set them describes the network there, and that each layer's numbers are divided among their sources by the authors' rule.
7. **What does it not show?**
   - **What removing a connection would do.** Step 5 shows the two can rank connections quite differently.
   - **That the graph is useful.** Walks describe one trained network; comparing graphs needs retraining.
   - **Anything causal about engines**, or cycles: the shares are parts of the score.

**Do not confuse** a large walk share with an important connection: shares are large and cancel, and they are partly fixed by the graph's own weights before training.

## 6. Final verification

In [ ]:
checks = [
    ("The loaded network scores what the run recorded", same_network, f"seed {SEED}"),
    ("Walks plus constant give the score on the worked window", adds_up,
     f"{conservation_error(result):.1e}"),
    ("The score through the sigmoid is the prediction", turns_into_prediction, f"engine {engine}"),
    ("At gamma 0, each sensor's total equals the gradient check", gi_matches_gradient, "14 sensors"),
    ("Walks plus constant give the score at every window", bool(conservation_error(life) < 1e-9),
     f"{conservation_error(life):.1e} over {len(engine_rows)} windows"),
    ("The check's full graph scores what the walks split", full_graph_scores, "5 windows"),
]
width = max(len(name) for name, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 40))
for name, passed, detail in checks:
    print(f"{name:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")
print("-" * (width + 40))
failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")

## 7. Summary

### What this notebook produced

- **Walk shares** for one window under the authors' setting, checked against the score, against PyTorch's own differentiation at γ = 0, and against the network's prediction.
- **The engine's whole life**: what stays within a group and what moves, and what flowed along each connection of the graph.
- **The authors' own check** on every window of the engine, beside simpler and random orders, and the stored five-seed measurement with each network's untrained starting point.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| Explaining walks at all | GNN-LRP, not in the paper, on the score before the sigmoid, with the authors' own setting of γ and 0 at the first graph layer | A-29 |
| Running the authors' check | a sensor taken out by zeroing its row and column of the rescaled graph; 20 random orders | A-33 |
| What removing a connection means | deleted from the raw graph both ways, graph rescaled, network not retrained | A-30 |
| Which engine is followed over its life | the held-back engine with the most windows, as in every explanation notebook here | this notebook |

### References

- Schnake, T., Eberle, O., Lederer, J., Nakajima, S., Schütt, K. T., Müller, K.-R., & Montavon, G. (2022). Higher-order explanations of graph neural networks via relevant walks. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 44(11), 7581-7596.